In [31]:
!pip install pydantic
!pip install fastapi
from pydantic import BaseModel
import joblib
import pandas as pd
from fastapi import FastAPI

In [12]:
# checking curr working directory .a
# import os
# print(os.getcwd())

Creating instance for fastAPI

In [32]:
app = FastAPI(title="Bank churn prediction API")

importing model and model cols from joblib

In [33]:

model = joblib.load("../models/bank_churn_model.pkl")
model_cols = joblib.load("../models/model_columns.pkl")


Creating a customer data class which is given to the API by user 

In [34]:
class CustomerData(BaseModel):
    CreditScore :int
    Geography:str
    Gender : str
    Age :int
    Tenure:int
    Balance:float
    NumOfProducts:int
    HasCrCard:int
    IsActiveMember:int 
    EstimatedSalary:float

Preprocessing input data for campatibility with how model takes data.  

In [35]:
def preprocess_data(data:CustomerData):
    # first convert coming data as dictionary
    input_dict = data.dict()

    # converting input dict to pandas dataframe
    df = pd.DataFrame([input_dict])

    # applying one - hot encoding to the dataFrame just like model
    df = df.get_dummies(df,columns = ['Geography','Gender'],drop_first = True)

    # make sure ALL expected columns exist, even if this request didn't produce them
    for col in model_cols:
        if col not in df.columns:
            df[col] = 0 
            
    # reorder columns to EXACTLY match what the model was trained on
    df = df[model_cols]
    
    return df
    

Creating API routes 

In [36]:
@app.get('/')
def root():
    return {'message':'Churn prediction api is running...'}

In [37]:
@app.get('/health')
def health():
    return {'status':'ok'}

In [38]:
@app.get('/predict')
def predict(data:CustomerData):
    df = preprocess_data(data)

    predict_prob = model.predict_proba(df)[0][1]
    prediction = int(predict_prob > 0.5 )

    return {
        'churn_probablity': round(float(predict_prob),4),
        'churn_prediction' : prediction,
        'risk_level' : "High" if predict_prob > 0.5 else "Low"
    }